### Imports

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
from shapely.ops import unary_union
from shapely.geometry import Polygon
from shapely.ops import nearest_points

### --- 1. Load Data ---

In [ ]:
# Set up base path and crs
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")
jamaica_metric_grid_crs = "EPSG:3448"

# Load the Jamaica boundary (optional, for context)

jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)

print(jamaica_boundary.crs)

# Load land use data

land_use = base_path / "2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use)

# Reproject to Jamaica Metric Grid (EPSG:3448)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print("Landcover CRS:", terrestrial_landcover.crs)

### --- 2. Filter to Forest Patches -------

In [ ]:
# 2. Define which classes count as "Forest"
#    Adjust these strings to match your actual data's classification
forest_classes = [
    'Open dry forest - Short',
    'Open dry forest - Tall (Woodland/Savanna)',
    'Disturbed broadleaved forest (Secondary Forest)',
    'Closed broadleaved forest (Primary Forest)',
    'Secondary Forest',
]


In [ ]:
# 3. Filter all forest polygons
all_forests = terrestrial_landcover[terrestrial_landcover["Classify"].isin(forest_classes)].copy()
print(f"Number of forest polygons: {len(all_forests)}")

In [ ]:
# 4. Optional: Fix invalid geometries
all_forests["geometry"] = all_forests.geometry.buffer(0)
print("Valid geometries:", all_forests.is_valid.sum(), "/", len(all_forests))

###  2. Analyze Patches, Distances, or Connectivity

In [ ]:
# Depending on your goal, you can:
	#•	(A) Calculate area and rank each forest polygon.
	#•	(B) Merge or union them to measure how many forest patches exist.
	#•	(C) Calculate the nearest patch for each polygon.


In [ ]:
# # Example A: Area & Rank

# Calculate area
all_forests["area_m2"] = all_forests.geometry.area

# Rank patches by area
all_forests["area_rank"] = all_forests["area_m2"].rank(method="dense", ascending=False)

print(all_forests[["Classify", "area_m2", "area_rank"]].head())

In [ ]:
# Example B: Count How Many Forest Patches

from shapely.ops import unary_union

# Combine all forest polygons into a single geometry
forest_union = unary_union(all_forests.geometry)

def count_patches(geom):
    """Returns how many sub-polygons exist in this union geometry."""
    if geom.is_empty:
        return 0
    if geom.geom_type == "Polygon":
        return 1
    elif geom.geom_type == "MultiPolygon":
        return len(list(geom.geoms))
    return 0

num_patches = count_patches(forest_union)
print(f"Number of contiguous forest patches: {num_patches}")

In [ ]:
# Suppose 'all_forests' is your GeoDataFrame with 21k polygons
# or "primary_broadleaved_forest_patches" / "terrestrial_landcover", etc.

sindex = all_forests.sindex

def find_nearest_patch_and_distance(geom, gdf, sindex):
    """
    Returns:
      - nearest_patch_id (int index in gdf, or NaN)
      - dist (float distance in CRS units, or NaN)
    """
    # **Unpack two arrays**: left_idxs (for the query), right_idxs (for the matches)
    left_idxs, right_idxs = sindex.nearest([geom], return_all=False)
    
    # If no match found
    if len(right_idxs) == 0:
        return np.nan, np.nan
    
    # There's exactly one match for a single geometry query
    match_idx = right_idxs[0]
    
    # Retrieve the matched polygon from gdf
    candidate = gdf.iloc[match_idx]
    
    # Optionally exclude self-match (if the candidate is the same polygon)
    if candidate.geometry.equals_exact(geom, tolerance=0):
        # No meaningful nearest patch if it's the same geometry
        return np.nan, np.nan
    
    # Calculate the distance
    nearest_geom = nearest_points(geom, candidate.geometry)[1]
    dist = geom.distance(nearest_geom)
    
    return match_idx, dist

# 3. Loop through polygons, storing nearest patch info
nearest_patch_ids = []
nearest_dists = []

# (Optional) slight performance boost by converting geometry and index to lists
geom_list = all_forests.geometry.tolist()
index_list = all_forests.index.tolist()

for idx, geom in zip(index_list, geom_list):
    patch_id, dist = find_nearest_patch_and_distance(geom, all_forests, sindex)
    nearest_patch_ids.append(patch_id)
    nearest_dists.append(dist)
    
# 4. Save to new columns
all_forests["nearest_patch_id"] = nearest_patch_ids
all_forests["nearest_dist_m"]   = nearest_dists

# 5. Inspect results
print(all_forests[["nearest_patch_id", "nearest_dist_m"]].head())

## 1. Identify “Afforestable” Areas


In [ ]:
# Just an example -- replace with your actual classes that you can convert to forest
afforestable_classes = [
    "Fields: Bare Land",
    "Quarry",
    "Bauxite Extraction",
    "Bamboo",
    "Fields: Herbaceous crops, fallow, cultivated vegetables",
    "Fields: Pasture,Human disturbed, grassland",
    "Bamboo and Fields",
    "Fields  and Bamboo",
    ]

# Filter these polygons from your land use GeoDataFrame
afforestable_areas = terrestrial_landcover[
    terrestrial_landcover["Classify"].isin(afforestable_classes)
].copy()

# Fix invalid geometries, if needed
afforestable_areas["geometry"] = afforestable_areas.geometry.buffer(0)

print(f"Number of afforestable polygons: {len(afforestable_areas)}")

### 2. Baseline Connectivity of Current Forest

In [ ]:
from shapely.ops import unary_union

# Union of all existing forest polygons
forest_union = unary_union(all_forests.geometry)

def count_patches(geom):
    """Returns how many sub-polygons (patches) are in this union geometry."""
    if geom.is_empty:
        return 0
    if geom.geom_type == "Polygon":
        return 1
    elif geom.geom_type == "MultiPolygon":
        return len(list(geom.geoms))
    return 0

baseline_patch_count = count_patches(forest_union)
print("Baseline # of forest patches:", baseline_patch_count)

### 3. Test Each Afforestable Polygon (Scenario Approach)

In [ ]:
import numpy as np

results = []

for idx, row in afforestable_areas.iterrows():
    # "Afforest" by unioning this polygon with the existing forest union
    scenario_union = forest_union.union(row.geometry)
    
    # Count how many patches remain in this scenario
    scenario_patch_count = count_patches(scenario_union)
    
    # Improvement is the difference from baseline
    improvement = baseline_patch_count - scenario_patch_count
    
    results.append({
        "afforestable_idx": idx,
        "land_class": row["Classify"],
        "patch_count_after": scenario_patch_count,
        "patch_improvement": improvement
    })

# Convert results to a DataFrame
results_df = pd.DataFrame(results).sort_values("patch_improvement", ascending=False)
print(results_df.head(10))

### 4. Rank or Select Top Afforestation Candidates

In [ ]:
top_candidates = results_df[results_df["patch_improvement"] > 0]
print("Top polygons that actually merge forest patches:")
print(top_candidates.head(10))

### 5. Visualize a Few Key Candidates

In [ ]:
import matplotlib.pyplot as plt

# Suppose you pick the best single candidate
best_idx = top_candidates.iloc[0]["afforestable_idx"]

fig, ax = plt.subplots(figsize=(10, 8))

# Plot existing forest polygons
all_forests.plot(ax=ax, color="green", edgecolor="black", alpha=0.5, label="Existing Forest")

# Plot the "best" afforestable polygon
best_poly = afforestable_areas.loc[best_idx]
gpd.GeoDataFrame(pd.DataFrame([best_poly]), geometry="geometry").plot(
    ax=ax, color="red", edgecolor="black", alpha=0.7, label="Best Afforest Polygon"
)

ax.set_title("Afforestation Scenario: Merging Forest Patches")
ax.legend()
plt.show()

In [ ]:
# # Example C: Nearest Patch Analysis
# all_forests["nearest_patch_id"] = np.nan
# all_forests["nearest_dist_m"] = np.nan

# for i, row in all_forests.iterrows():
#     others = all_forests.drop(i)
#     dist_series = others.geometry.distance(row.geometry)
    
#     min_dist = dist_series.min()
#     min_idx = dist_series.idxmin()
    
#     all_forests.at[i, "nearest_dist_m"] = min_dist
#     all_forests.at[i, "nearest_patch_id"] = min_idx

# # Inspect
# print(all_forests[["Classify", "nearest_patch_id", "nearest_dist_m"]].head())

In [ ]:
# # 3. Plot the Results
# fig, ax = plt.subplots(figsize=(10, 8))

# # Optional context: plot the Jamaica boundary
# jamaica_boundary.plot(ax=ax, color="lightgrey", edgecolor="black")

# # Plot forest patches, colored by nearest distance
# all_forests.plot(
#     column="nearest_dist_m",
#     cmap="viridis",
#     legend=True,
#     edgecolor="black",
#     alpha=0.8,
#     ax=ax
# )

# # Draw lines from each patch to its nearest patch
# for i, row in all_forests.iterrows():
#     if pd.isna(row["nearest_patch_id"]):
#         continue
    
#     x1, y1 = row.geometry.centroid.x, row.geometry.centroid.y
#     nearest_id = int(row["nearest_patch_id"])
#     nearest_row = all_forests.loc[nearest_id]
    
#     x2, y2 = nearest_row.geometry.centroid.x, nearest_row.geometry.centroid.y
#     ax.plot([x1, x2], [y1, y2], color="red", linewidth=1, alpha=0.7)

# ax.set_title("Nearest Distances Among All Forest Types", fontsize=14)
# plt.show()